# 03 · Anomaly experiments — does the hybrid catch what supervision misses?

In [1]:
import pandas as pd

pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 20)

from threatsense.common.config import get_settings
from threatsense.storage.database import Database

db = Database(get_settings().db_path)
gm = db.query_df("SELECT * FROM group_metrics WHERE grouping='category' AND grp != 'benign'")
gm.pivot(index="grp", columns="model", values="recall").round(3)

model,hist_gradient_boosting,hybrid,isolation_forest,logistic_regression,majority_class,random_forest,zscore_threshold
grp,,,,,,,
botnet,1.000,1.0,0.000,1.0,0.0,1.0,0.001
bruteforce,1.000,1.0,0.296,1.0,0.0,1.0,0.750
dos,1.000,1.0,0.997,1.0,0.0,1.0,1.000
exfiltration,0.292,1.0,0.992,0.0,0.0,0.4,0.575
portscan,1.000,1.0,1.000,1.0,0.0,1.0,0.991


## Novel attack: exfiltration (absent from training)

In [2]:
flows = db.query_df("SELECT category, supervised_proba, anomaly_pct, risk, alert FROM scored_flows")
flows.groupby("category")[["supervised_proba", "anomaly_pct", "risk", "alert"]].mean().round(3)

,supervised_proba,anomaly_pct,risk,alert
category,,,,
benign,0.004,0.499,0.153,0.012
botnet,0.998,0.719,0.914,1.000
bruteforce,0.988,0.940,0.974,1.000
dos,0.999,1.000,0.999,1.000
exfiltration,0.011,0.999,0.307,1.000
portscan,0.998,1.000,0.999,1.000


**Finding.** The supervised model was never shown exfiltration and scores many of those flows low.
The Isolation Forest — trained on benign traffic only — ranks them as extreme outliers (≈0.999
percentile), and the hybrid risk lifts them *just* above the frozen threshold (mean risk ≈0.31 vs
threshold ≈0.30). That margin is fragile: a weighted average dilutes a strong single signal.
**Next experiment:** max-fusion (alert if either component exceeds its own validated threshold).
The test FPR (≈1.2 %) is slightly above the 1 % validation budget — reported, not tuned away.